# 02 — Inference

Quick walkthrough of running the persisted Heart Disease pipeline on (a) a single record and (b) a small batch from the held-out test set.

The whole point is that nothing here re-trains the model. We just load `models/heart_disease_model.pkl` and call `.predict` / `.predict_proba` on the same `ColumnTransformer + classifier` pipeline that the FastAPI service uses in production.

Run order: `01_eda.ipynb` → `python -m src.models.train` → this notebook.

In [1]:
import sys
from pathlib import Path

# Make `src.*` importable when running the notebook from `notebooks/`.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import joblib
import pandas as pd
from src.config import FEATURE_COLUMNS, MODEL_PATH, TARGET_LABELS, TEST_CSV

MODEL_PATH, TEST_CSV

(PosixPath('/Users/ramyks/Downloads/MLOps_Assignment/heart_disease_mlops/models/heart_disease_model.pkl'),
 PosixPath('/Users/ramyks/Downloads/MLOps_Assignment/heart_disease_mlops/data/processed/test.csv'))

## 1. Load the pipeline

In [2]:
pipeline = joblib.load(MODEL_PATH)
type(pipeline).__name__, [s[0] for s in pipeline.steps]

('Pipeline', ['preprocessor', 'classifier'])

## 2. Single-record prediction
A high-risk profile from the dataset (older male, chest-pain type 4 etc.). The pipeline takes the raw 13 features — scaling and one-hot encoding happen inside it.

In [3]:
patient = {
    "age": 67, "sex": 1, "cp": 3, "trestbps": 160, "chol": 286,
    "fbs": 0, "restecg": 2, "thalach": 108, "exang": 1, "oldpeak": 1.5,
    "slope": 2, "ca": 3, "thal": 3,
}
X_one = pd.DataFrame([patient])[FEATURE_COLUMNS]
pred = int(pipeline.predict(X_one)[0])
proba = pipeline.predict_proba(X_one)[0]
print(f"prediction: {pred} ({TARGET_LABELS[pred]})")
print(f"P(no_disease)={proba[0]:.4f}  P(disease)={proba[1]:.4f}")

prediction: 1 (disease)
P(no_disease)=0.2422  P(disease)=0.7578


## 3. Batch prediction on the held-out test set
Useful sanity check: confidence should track correctness on average — i.e. when the model is wrong, it tends to be less sure.

In [4]:
test_df = pd.read_csv(TEST_CSV)
X_test = test_df[FEATURE_COLUMNS]
y_test = test_df["target"].astype(int)

preds = pipeline.predict(X_test)
probas = pipeline.predict_proba(X_test)
out = pd.DataFrame({
    "y_true": y_test,
    "y_pred": preds.astype(int),
    "confidence": probas.max(axis=1).round(4),
    "P(disease)": probas[:, 1].round(4),
})
out.head(10)

,y_true,y_pred,confidence,P(disease)
0,0,0,0.9746,0.0254
1,0,0,0.9417,0.0583
2,0,0,0.5360,0.4640
3,0,0,0.5786,0.4214
4,0,0,0.6801,0.3199
5,0,0,0.9883,0.0117
6,1,1,0.8286,0.8286
7,0,0,0.9390,0.0610
8,1,0,0.7514,0.2486
9,0,0,0.9437,0.0563


In [5]:
# Mean confidence on correct vs incorrect predictions.
out["correct"] = (out["y_true"] == out["y_pred"]).astype(int)
out.groupby("correct")["confidence"].agg(["count", "mean", "std"]).round(4)

,count,mean,std
correct,,,
0,11,0.6674,0.1461
1,49,0.9009,0.1192


## 4. The same call, but through the HTTP API
Equivalent to step 2; this is what `scripts/predict.py` and the FastAPI `/predict` endpoint both end up doing under the hood. Run the API first:

```bash
make serve   # uvicorn on :8080
# or
docker run -d -p 8080:8080 ghcr.io/ks-ramya/heart-disease-api:latest
```

Then:

```bash
curl -s -X POST http://127.0.0.1:8080/predict \
  -H 'Content-Type: application/json' \
  -d '{"age":67,"sex":1,"cp":3,"trestbps":160,"chol":286,"fbs":0,"restecg":2,\
       "thalach":108,"exang":1,"oldpeak":1.5,"slope":2,"ca":3,"thal":3}'
```